# 🛣️ Lane Detection Training — Vietnam ADAS
## Multi-class Lane Segmentation cho hệ thống hỗ trợ lái xe (ADAS)

**Mục tiêu**: Train mô hình `YOLO11n-seg` phát hiện **11 loại vạch/lane** đặc thù đường Việt Nam, phục vụ cảnh báo:
- Đè vạch liền (không được đè)
- Đè vạch đứt (có thể chuyển làn)
- Lệch làn (lane departure)
- Vạch dừng, vạch qua đường, mũi tên chỉ hướng...

**Model**: `YOLO11n-seg` (Ultralytics) — nhẹ, real-time, tối ưu cho edge device (Jetson).  
**Backbone cache**: Tải từ Hugging Face và lưu cache trên Google Drive để không mất khi session Colab disconnect.

### Pipeline tổng quan
1. Mount Google Drive + cài đặt môi trường
2. Tải `yolo11n-seg.pt` và dataset lane từ Hugging Face → cache Drive
3. Chuẩn hoá & merge nhiều dataset lane (VIA, BDD100K, TuSimple, ZOD-Mini) → 11 classes chuẩn VN
4. Train `YOLO11n-seg` với augmentation cho đường Việt
5. Validate (mAP, mIoU, Precision, Recall) + visualize predictions
6. Export ONNX / TorchScript → sẵn sàng deploy ADAS

### 11 Classes (chuẩn ADAS Việt Nam)
| ID | Class | Mô tả | Màu (BGR) |
|---|---|---|---|
| 0 | `background` | Nền (không phải lane/marking) | (0,0,0) |
| 1 | `road` | Mặt đường (drivable area) | (128,128,128) |
| 2 | `lane_white_solid` | Vạch trắng liền — KHÔNG được đè | (255,255,255) |
| 3 | `lane_white_dashed` | Vạch trắng đứt — được phép chuyển làn | (200,200,255) |
| 4 | `lane_yellow_solid` | Vạch vàng liền — KHÔNG được đè (rất nguy hiểm) | (0,255,255) |
| 5 | `lane_yellow_dashed` | Vạch vàng đứt — được phép vượt cẩn thận | (0,200,200) |
| 6 | `lane_double_yellow` | Vạch vàng đôi — cấm tuyệt đối | (0,150,150) |
| 7 | `stop_line` | Vạch dừng (stop line) | (0,0,255) |
| 8 | `crosswalk` | Vạch sang đường (crosswalk/zebra) | (255,0,255) |
| 9 | `arrow_straight` | Mũi tên thẳng | (0,255,0) |
| 10 | `arrow_turn` | Mũi tên rẽ (trái/phải) | (255,128,0) |

---

**Tham số chính sẽ hiển thị sau khi train**:
- `mAP50(box)` / `mAP50(mask)`
- `mAP50-95(box)` / `mAP50-95(mask)`
- `Precision` / `Recall`
- `mIoU` (mean Intersection-over-Union)
- `Inference latency` (ms/frame)
- `Model size` (MB) → để đánh giá deploy edge


## 📂 PHẦN 1 — Kết nối Google Drive

Mọi thứ (dataset cache, weights, kết quả training) sẽ được lưu trong thư mục `ADAS_Lane_VN/` trên Drive.
Nhờ đó khi Colab reconnect hay mở lại notebook sau, dữ liệu/weights **không bị mất**.

In [ ]:
import os, sys, time, json, shutil, pathlib, subprocess, importlib

# -------- Cấu hình thư mục gốc trên Drive --------
DRIVE_ROOT = "/content/drive/MyDrive/ADAS_Lane_VN"
DIRS = {
    "root":      DRIVE_ROOT,
    "models":    f"{DRIVE_ROOT}/models",         # cache .pt tải từ Hugging Face
    "datasets":  f"{DRIVE_ROOT}/datasets",       # dataset lane cache
    "weights":   f"{DRIVE_ROOT}/weights",        # weights sau train
    "runs":      f"{DRIVE_ROOT}/runs",           # log / kết quả training
    "exports":   f"{DRIVE_ROOT}/exports",        # model xuất ra (onnx, torchscript)
    "hf_cache":  f"{DRIVE_ROOT}/hf_cache",       # cache huggingface_hub
    "logs":      f"{DRIVE_ROOT}/logs",           # log JSON / metrics
}
for k, p in DIRS.items():
    os.makedirs(p, exist_ok=True)
    print(f"[DIR] {k:9s} -> {p}")

# -------- Mount Drive (idempotent — nếu đã mount thì bỏ qua) --------
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("[MOUNT] Drive vừa được mount.")
else:
    print("[MOUNT] Drive đã có sẵn, bỏ qua mount.")

# -------- Đặt biến môi trường cho cache --------
os.environ["HF_HOME"]             = DIRS["hf_cache"]
os.environ["HUGGINGFACE_HUB_CACHE"] = DIRS["hf_cache"]
os.environ["TRANSFORMERS_CACHE"]  = DIRS["hf_cache"]
os.environ["TORCH_HOME"]          = f"{DIRS['models']}/torch_hub"
os.environ["YOLO_CONFIG_DIR"]      = f"{DIRS['root']}/ultralytics_cfg"
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)
os.makedirs(os.environ["YOLO_CONFIG_DIR"], exist_ok=True)

print("\n[ENV] Biến môi trường đã đặt xong — mọi cache sẽ nằm trên Drive.")
for v in ["HF_HOME", "HUGGINGFACE_HUB_CACHE", "TORCH_HOME", "YOLO_CONFIG_DIR"]:
    print(f"  {v} = {os.environ[v]}")

## 📦 PHẦN 2 — Cài đặt thư viện

Cài `ultralytics` (YOLO11), `huggingface_hub`, `roboflow` (cho dataset BDD/TuSimple), `opencv`, `albumentations` (augmentation mạnh cho đường Việt).

Phiên bản được pin cụ thể để tránh break khi Colab nâng cấp runtime.

In [ ]:
import subprocess, sys, importlib

def pip_install(pkg):
    print(f"[pip] {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

REQUIRED = [
    "ultralytics==8.3.40",            # YOLO11
    "huggingface_hub==0.27.1",        # tải dataset/model từ HF
    "datasets==3.2.0",                # HF datasets loader
    "albumentations==1.4.24",         # augmentation
    "opencv-python-headless==4.10.0.84",
    "roboflow==1.1.50",               # dataset BDD / TuSimple qua roboflow API
    "onnx==1.17.0",
    "onnxruntime==1.19.2",
    "tqdm==4.67.1",
    "pyyaml==6.0.2",
    "requests==2.32.3",
    "hf_transfer==0.1.6",             # tải nhanh từ HF
]
for p in REQUIRED:
    pip_install(p)

print("\n[OK] Đã cài xong toàn bộ thư viện.")

# ---- Verify version ----
import ultralytics, huggingface_hub, datasets, cv2, albumentations, onnx, onnxruntime
print(f"ultralytics        : {ultralytics.__version__}")
print(f"huggingface_hub    : {huggingface_hub.__version__}")
print(f"datasets           : {datasets.__version__}")
print(f"opencv             : {cv2.__version__}")
print(f"albumentations     : {albumentations.__version__}")
print(f"onnx / onnxruntime : {onnx.__version__} / {onnxruntime.__version__}")

## 🔐 PHẦN 3 — Đăng nhập Hugging Face & kiểm tra GPU

Dùng `HF_TOKEN` bạn cung cấp để tải dataset/model (một số repo yêu cầu `gated=true` hoặc tốc độ nhanh hơn).
Token sẽ được lưu vào `~/.huggingface/token` để Colab session sau không phải nhập lại.

In [ ]:
import os, json, re, torch, requests, subprocess
from huggingface_hub import login, whoami

# ============================================================
# API CONFIG & LIVE VALIDATION
# Secrets được đọc từ file .env (token KHÔNG được commit vào git)
# ============================================================
print("[API] Dang setup 3 nguon dataset (HF + Kaggle + Roboflow)...\n")

# Load .env
try:
    from dotenv import load_dotenv
    import pathlib
    dotenv_path = pathlib.Path('/Users/melaniepham/Documents/Viet/2026/xử lý ảnh/DA/ADAS-MOI/ADAS/.env')
    if dotenv_path.exists():
        load_dotenv(dotenv_path)
        print('[ENV] Loaded secrets from .env')
    else:
        print('[ENV] .env not found — reading from environment')
except ImportError:
    print('[ENV] python-dotenv not installed — reading from environment')

# --- HuggingFace ---
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
try:
    if os.environ["HF_TOKEN"]:
        login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
        user = whoami(token=os.environ["HF_TOKEN"])
        print(f"[HF]      OK    : {user.get('name','?')} ({user.get('fullname','?')})")
    else:
        print('[HF]      SKIP  : HF_TOKEN not set')
except Exception as e:
    print(f"[HF]      FAIL  : {e}")

# ============================================================
# Kaggle
# ============================================================
os.environ["KAGGLE_USERNAME"] = os.environ.get("KAGGLE_USERNAME", "")
os.environ["KAGGLE_KEY"]      = os.environ.get("KAGGLE_KEY", "")

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
if os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_KEY"]:
    with open(kaggle_json_path, "w") as f:
        json.dump({
            "username": os.environ["KAGGLE_USERNAME"],
            "key":      os.environ["KAGGLE_KEY"],
        }, f, indent=2)
    os.chmod(kaggle_json_path, 0o600)
    print(f"[KAGGLE]  OK    : username={os.environ['KAGGLE_USERNAME']}")
else:
    print('[KAGGLE]  SKIP  : KAGGLE_USERNAME or KAGGLE_KEY not set')

try:
    r = subprocess.run(["kaggle", "datasets", "list"], capture_output=True, text=True, timeout=20)
    if r.returncode == 0:
        lines = r.stdout.strip().splitlines()
        print(f"[KAGGLE-TEST] OK : {len(lines)} datasets")
    else:
        print(f"[KAGGLE-TEST] FAIL rc={r.returncode}")
except FileNotFoundError:
    print("[KAGGLE-TEST] CLI chua cai - se install o Cell 10")
except Exception as e:
    print(f"[KAGGLE-TEST] ERR: {e}")

# --- Roboflow ---
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

rf_status = "UNKNOWN"; rf_msg = ""
if ROBOFLOW_API_KEY:
    try:
        r = requests.post(
            "https://api.roboflow.com/?api_key=" + ROBOFLOW_API_KEY,
            timeout=10
        )
        if r.status_code == 200:
            data = r.json()
            rf_status = "OK"
            rf_msg = f"workspace: {data.get('workspace','?')}"
        elif r.status_code == 401:
            rf_status = "FAIL_401"
            rf_msg = "Key bi tu choi. Check lai Private key."
        else:
            rf_status = f"FAIL_{r.status_code}"
            rf_msg = r.text[:120]
    except Exception as e:
        rf_status = "NETWORK_FAIL"
        rf_msg = str(e)[:120]
else:
    rf_status = "SKIP"
    rf_msg = "ROBOFLOW_API_KEY not set"

print(f"[ROBOFLOW] {rf_status:12s} : {rf_msg}")


## ⬇️ PHẦN 4 — Tải model `yolo11n-seg.pt` từ Hugging Face (cache Drive)

Model `yolo11n-seg.pt` được tải về `DIRS['models']`. Nếu đã có (do session trước để lại) thì bỏ qua tải lại.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil, os, time, requests

# ============================================================
# Guard DIRS (neu quen chay Cell 7)
# ============================================================
try:
    MODEL_DIR = Path(DIRS["models"])
    WEIGHTS_DIR = Path(DIRS["weights"])
    print(f"[PATHS] DIRS OK (Cell 7 da chay)")
except NameError:
    print("[FIX] DIRS chua co - auto-mount Drive...")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    BASE = Path("/content/drive/MyDrive/adas_project")
    BASE.mkdir(parents=True, exist_ok=True)
    DIRS = {
        "datasets":   BASE / "datasets",
        "models":     BASE / "models",
        "weights":    BASE / "weights",
        "runs":       BASE / "runs",
        "exports":    BASE / "exports",
        "logs":       BASE / "logs",
        "final_model": BASE / "final_model",
        "data_yaml":  BASE / "data_final.yaml",
    }
    for k, p in DIRS.items():
        Path(p).mkdir(parents=True, exist_ok=True)
        print(f"  {k:12s} = {p}")
    MODEL_DIR  = Path(DIRS["models"])
    WEIGHTS_DIR = Path(DIRS["weights"])

MODEL_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

def download_url(url, target):
    print(f"[FETCH URL] {url}")
    t0 = time.time()
    with requests.get(url, stream=True, timeout=180, allow_redirects=True) as r:
        r.raise_for_status()
        with open(target, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk: f.write(chunk)
    print(f"[OK] {target.name} ({target.stat().st_size/1e6:.1f} MB) trong {time.time()-t0:.1f}s")

def hf_try(repo_id, filename, target):
    try:
        p = hf_hub_download(
            repo_id=repo_id, filename=filename, repo_type="model",
            local_dir=str(target.parent), token=os.environ.get("HF_TOKEN"),
        )
        if Path(p) != target:
            shutil.copy2(p, target)
        print(f"[HF OK] {repo_id}/{filename} -> {target}")
        return True
    except Exception as e:
        print(f"[HF FAIL] {repo_id}: {type(e).__name__}: {str(e)[:120]}")
        return False

# Lay yolo11n-seg.pt voi 4 lop fallback (uu tien theo thu tu)
w_dst = WEIGHTS_DIR / "yolo11n-seg.pt"
yolo_seg_path = w_dst

if w_dst.exists() and w_dst.stat().st_size > 1024 * 1024:
    print(f"[CACHE] {w_dst.name} da co san tren Drive ({w_dst.stat().st_size/1e6:.1f} MB), bo qua tai.")
else:
    # (1) GitHub release chinh thuc cua Ultralytics - nguon goc, nhanh, khong bi 403
    try:
        download_url(
            "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n-seg.pt",
            w_dst,
        )
    except Exception as e:
        print(f"[URL FAIL] {e}")
        # (2) Hugging Face mirror do cong dong upload
        if not hf_try("Bingsu/yolo11n-seg", "yolo11n-seg.pt", w_dst):
            # (3) Repo Ultralytics khac tren HF
            if not hf_try("Ultralytics/yolo11", "yolo11n-seg.pt", w_dst):
                # (4) Cuoi cung de ultralytics tu tai ve cwd
                from ultralytics import YOLO
                tmp = YOLO("yolo11n-seg.pt")
                shutil.copy2("yolo11n-seg.pt", w_dst)
                print(f"[ULTRALYTICS AUTO] -> {w_dst}")

print(f"\n[MODEL] yolo11n-seg.pt = {yolo_seg_path}")

# Backup copy vao cache de lan sau dung nhanh
cache_copy = MODEL_DIR / "yolo11n-seg.pt"
if not cache_copy.exists():
    shutil.copy2(yolo_seg_path, cache_copy)
    print(f"[COPY] -> {cache_copy}")

## 📚 PHẦN 5 — Tải Dataset (đầy đủ 11 classes)

Chien luoc: gop nhieu nguon dataset de co đầy đủ 11 classes chuan VN.

| # | Nguon | Class dat duoc | Luong |
|---|---|---|---|
| 1 | **VIA Lane Segment** (GitHub) | `road`(1) + `lane_white_*`(2,3) | ~6k anh VN that |
| 2 | **ZOD-Mini 2D** (HF) | `road` augment | ~2k |
| 3 | **Kaggle: road-marking-yolov8** | `solid`/`dashed` | ~1.5k |
| 4 | **Kaggle: road-lane-segmentation** | `lane` mask  | ~3k |
| 5 | **Kaggle: crosswalk-detection** | `crosswalk` (8) | ~500 |
| 6 | **Kaggle: traffic-sign-yolo** | `arrow` (9,10) | ~1k |

Sau khi tai, ta se:
1. Map moi dataset ve **11 classes chuan VN** (xem Cell 12-13)
2. Gop vao `/content/drive/MyDrive/ADAS_Lane_VN/datasets/merged_lane_vn/`
3. Sinh `lane_vn.yaml` (11 classes)

> **Luu y**: Tat ca Kaggle datasets o tren deu public (khong can API key dac biet).

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os, time, requests, zipfile, subprocess, shutil, sys, json
from tqdm.auto import tqdm

# ============================================================
# DIRS guard
# ============================================================
try:
    DATA_ROOT = Path(DIRS["datasets"])
    MODELS_DIR = Path(DIRS["models"])
    print(f"[PATHS] DIRS OK")
except NameError:
    print("[FIX] DIRS chua co - auto-mount Drive...")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    BASE = Path("/content/drive/MyDrive/adas_project")
    BASE.mkdir(parents=True, exist_ok=True)
    DIRS = {
        "datasets":   BASE / "datasets",
        "models":     BASE / "models",
        "runs":       BASE / "runs",
        "exports":    BASE / "exports",
        "logs":       BASE / "logs",
        "final_model": BASE / "final_model",
        "data_yaml":  BASE / "data_final.yaml",
    }
    for k, p in DIRS.items():
        Path(p).mkdir(parents=True, exist_ok=True)
    DATA_ROOT  = Path(DIRS["datasets"])
    MODELS_DIR = Path(DIRS["models"])

DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# KAGGLE CLI - cai va test ky hon
# ============================================================
print("\n" + "="*70)
print("  [KAGGLE CLI] Setup")
print("="*70)

KAGGLE_OK = False
try:
    r = subprocess.run(["kaggle", "--version"], capture_output=True, text=True, timeout=15)
    if r.returncode == 0:
        print(f"[OK] da co: {r.stdout.strip()}")
        KAGGLE_OK = True
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("[...] dang cai dat kaggle...")

if not KAGGLE_OK:
    # Dam bao kaggle.json
    kaggle_dir  = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    if not kaggle_json.exists() and os.environ.get("KAGGLE_USERNAME"):
        kaggle_json.write_text(json.dumps({
            "username": os.environ["KAGGLE_USERNAME"],
            "key":       os.environ["KAGGLE_KEY"],
        }))
        kaggle_json.chmod(0o600)

    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kaggle==1.6.17"],
            check=True, capture_output=True, text=True
        )
        print("[OK] pip install kaggle")

        # Test lai
        r = subprocess.run(["kaggle", "--version"], capture_output=True, text=True, timeout=15)
        if r.returncode == 0:
            print(f"[OK] kaggle CLI san sang: {r.stdout.strip()}")
            KAGGLE_OK = True
        else:
            print(f"[FAIL] {r.stderr.strip()[:200]}")
    except Exception as e:
        print(f"[PIP FAIL] {e}")
        # Check binary path
        kp = subprocess.run([sys.executable, "-m", "kaggle", "--version"],
                            capture_output=True, text=True)
        print(f"  python -m kaggle: rc={kp.returncode}")
        if kp.returncode == 0:
            KAGGLE_OK = True

print(f"[KAGGLE_READY]: {KAGGLE_OK}")

# ============================================================
# HELPERS
# ============================================================
def hf_snapshot(repo_id, local_dir, allow_patterns=None):
    local_dir = Path(local_dir)
    if local_dir.exists() and any(local_dir.iterdir()):
        n = sum(1 for _ in local_dir.rglob("*") if _.is_file())
        if n > 10:
            print(f"[CACHE] HF {repo_id} ({n} files) - skip")
            return local_dir
    try:
        print(f"[HF] snapshot {repo_id}")
        snapshot_download(
            repo_id=repo_id, repo_type="dataset",
            local_dir=str(local_dir),
            allow_patterns=allow_patterns,
            token=os.environ.get("HF_TOKEN"),
        )
        print(f"[OK] HF {repo_id}")
    except Exception as e:
        print(f"[HF FAIL] {repo_id}: {type(e).__name__}: {str(e)[:200]}")
    return local_dir

def download_zip(url, dest_zip, dest_dir):
    dest_zip = Path(dest_zip); dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if any(dest_dir.iterdir()):
        print(f"[CACHE] {dest_dir.name} - skip"); return dest_dir
    print(f"[FETCH] {url}")
    try:
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(dest_zip, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest_zip.name) as bar:
                for chunk in r.iter_content(chunk_size=1<<20):
                    if chunk: f.write(chunk); bar.update(len(chunk))
        with zipfile.ZipFile(dest_zip) as z: z.extractall(dest_dir)
        print(f"[OK] {dest_zip.stat().st_size/1e6:.1f} MB")
    except Exception as e:
        print(f"[FAIL] {e}")
    return dest_dir

def kaggle_download(slug, dest_dir):
    """Tai Kaggle dataset - dung CLI, fallback SDK."""
    dest = Path(dest_dir)
    if dest.exists() and any(dest.iterdir()):
        print(f"[CACHE] Kaggle {slug} - skip")
        return dest
    if not KAGGLE_OK:
        print(f"[SKIP] kaggle CLI khong san sang - skip {slug}")
        return dest
    dest.mkdir(parents=True, exist_ok=True)

    # Lan 1: subprocess CLI
    print(f"[KAGGLE] {slug}")
    rc = subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(dest)],
        capture_output=True, text=True, timeout=600,
    ).returncode
    if rc != 0:
        print(f"[FALLBACK] thu Kaggle SDK...")
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
            api.dataset_download_files(slug, path=str(dest), unzip=True)
            print(f"[OK] {slug} (SDK)")
            return dest
        except Exception as e:
            print(f"[FAIL] {slug}: {e}")
            return dest

    # Unzip
    n_zip = 0
    for zf in list(dest.glob("*.zip")):
        try:
            with zipfile.ZipFile(zf) as z: z.extractall(dest)
            zf.unlink()
            n_zip += 1
        except Exception as e:
            print(f"  [UNZIP FAIL] {zf.name}: {e}")
    print(f"[OK] {slug} ({n_zip} zip)")
    return dest

def roboflow_download_universe(workspace, project, version, dest_dir):
    dest = Path(dest_dir)
    if dest.exists() and any(dest.iterdir()):
        print(f"[CACHE] Roboflow {project} - skip"); return dest
    rf_key = os.environ.get("ROBOFLOW_API_KEY", "")
    if not rf_key or rf_key in ("PASTE_PRIVATE_KEY_HERE", "Ge2oXXXXXXXXXXXXXXXX"):
        print(f"[SKIP] thieu ROBOFLOW_API_KEY"); return dest
    dest.mkdir(parents=True, exist_ok=True)
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=rf_key)
        rf.workspace(workspace).project(project).version(version).download(
            "yolov8", location=str(dest)
        )
        print(f"[OK] Roboflow {project}")
    except Exception as e:
        print(f"[ROBOFLOW FAIL] {project}: {e}")
    return dest

# ============================================================
# CHAY DOWNLOAD
# ============================================================
print("\n" + "="*70)
print("  BAT DAU TAI DATASET")
print("="*70)

# 1) VIA
print("\n[1/4] VIA Lane Segmentation")
download_zip(
    url="https://github.com/makerhanoi/via-dataset-cds-2020/releases/download/v1.0/Segmentation_Dataset.zip",
    dest_zip=DATA_ROOT / "via_seg.zip",
    dest_dir=DATA_ROOT / "via_lane_seg",
)

# 2) ZOD-Mini
print("\n[2/4] ZOD-Mini 2D (HF)")
hf_snapshot(
    repo_id="8bits-ai/ZOD-Mini-2D-Road-Scenes",
    local_dir=DATA_ROOT / "zod_mini",
    allow_patterns=["*.zip", "*.yaml", "*.md", "images/*", "labels/*"],
)

# 3) Kaggle datasets
print("\n[3/4] Kaggle datasets")
KAGGLE_LIST = [
    # Confirmed alive 2026-07-15
    ("dataclusterlabs/lane-detection-road-line-detection-image-dataset", "kaggle_lane_det_v1"),
    ("yaraeslam/enhanced-road-segmentation-dataset",                       "kaggle_enhanced_road"),
    # HuggingFace mirror (backup)
    ("veetinator/road-line-marking-dataset",                               "kaggle_rlmd"),
]
for slug, folder in KAGGLE_LIST:
    kaggle_download(slug, DATA_ROOT / folder)
    time.sleep(2)

# 4) Roboflow (optional - uncomment neu muon)
print("\n[4/4] Roboflow Universe (optional - SKIPED)")

print("\n" + "="*70)
print("  TONG KET CACHE")
print("="*70)
for p in sorted(DATA_ROOT.iterdir()):
    if p.is_dir():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"  {p.name:30s} : {n:6d} file")
print("="*70)

## 🛠️ PHẦN 6 — Chuẩn hoá & Merge dataset về **11 classes chuẩn VN**

Mỗi dataset nguồn có schema nhãn khác nhau (segmentation mask PNG, polygon YOLO, instance seg…).  
Ta sẽ viết bộ converter để:

1. **VIA** (`via_lane_seg/Segmentation/GGDataSet`): mask RGB → lớp lane_line (3), road (1), background (0)
2. **ZOD-Mini**: mask lane_marking → map sang `solid_line` / `dashed_line`
3. **alklibassy/road-marking**: parse `data.yaml` → map `solid`/`dashed` sang class 2/3 (white)
4. **ahmed-gamaleldin/lane**: bbox + class gốc → class 2

Tất cả gộp vào `/content/drive/MyDrive/ADAS_Lane_VN/datasets/merged_lane_vn/`

```
merged_lane_vn/
├── images/{train,val,test}/
├── labels/{train,val,test}/
└── lane_vn.yaml
```

In [ ]:
import cv2, json, yaml, shutil, random, numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter

MERGED = DATA_ROOT / "merged_lane_vn"
if MERGED.exists():
    shutil.rmtree(MERGED)
for sub in ["images/train", "images/val", "images/test",
            "labels/train", "labels/val", "labels/test"]:
    (MERGED / sub).mkdir(parents=True, exist_ok=True)

rng = random.Random(42)
stats = Counter()

# Helper: copy anh + ghi label YOLO-seg
def write_pair(img_path, label_lines, split):
    if not img_path.exists():
        return False
    stem = img_path.stem
    new_img = MERGED / f"images/{split}" / f"{stem}{img_path.suffix.lower()}"
    shutil.copy2(img_path, new_img)
    new_lbl = MERGED / f"labels/{split}" / f"{stem}.txt"
    new_lbl.write_text("\n".join(label_lines) if label_lines else "")
    return True

# =========================================================
# 1) VIA Lane Segmentation (RGB mask)
# =========================================================
# Tìm root GGDataSet robust (zip có thể giải nén ra /via_lane_seg/Segmentation/GGDataSet
# hoặc thẳng /via_lane_seg/GGDataSet)
def find_via_root():
    candidates = []
    for p in (DATA_ROOT / "via_lane_seg").rglob("GGDataSet"):
        if "__MACOSX" not in str(p):
            candidates.append(p)
    if not candidates:
        # fallback: tìm thư mục có *_frames/train
        for p in (DATA_ROOT / "via_lane_seg").rglob("*_frames"):
            if "__MACOSX" not in str(p):
                return p.parent  # parent là GGDataSet
        raise FileNotFoundError("Khong tim thay GGDataSet trong via_lane_seg/")
    return candidates[0]

VIA_ROOT = find_via_root()
print("[VIA] Root:", VIA_ROOT.relative_to(DATA_ROOT))

def via_split_paths(split):
    return (VIA_ROOT / f"{split}_frames" / split,  # frames/train/train_xxx.png
            VIA_ROOT / f"{split}_masks" / split)   # masks/train/train_xxx.png

# Palette ĐÚNG của VIA (xem via-dataset-cds-2020 repo):
#   road  = (128, 0, 0)   - đỏ sẫm
#   lane  = (192, 128, 128) - hồng nhạt
VIA_PALETTE = {
    "road": (128, 0, 0),
    "lane": (192, 128, 128),
}

def color_match_mask(img, target_rgb, tol=20):
    # Tim pixel gan target_rgb (BGR).
    target = np.array(target_rgb[::-1], dtype=int)
    diff = np.abs(img.astype(int) - target).sum(axis=2)
    return (diff < tol).astype(np.uint8) * 255

def via_mask_to_label(mask_path):
    # RGB -> polygon YOLO-seg lines: bg=0, road=1, lane=3.
    m = cv2.imread(str(mask_path))
    if m is None: return []
    H, W = m.shape[:2]
    out = []
    # cls_id map: road=1, lane white dashed (default cho via) = 3
    for cls_id, color in [(1, VIA_PALETTE["road"]), (3, VIA_PALETTE["lane"])]:
        mask = color_match_mask(m, color, tol=20)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in cnts:
            if cv2.contourArea(c) < 60: continue
            c = cv2.approxPolyDP(c, 0.003 * cv2.arcLength(c, True), True)
            if len(c) < 3: continue
            pts = c.reshape(-1, 2) / np.array([W, H])
            pts = np.clip(pts, 0.0, 1.0)
            line = f"{cls_id} " + " ".join(f"{x:.6f} {y:.6f}" for x, y in pts)
            out.append(line)
    return out

print("[MERGE] VIA Lane Segmentation ...")
via_count = 0
for split in ["train", "val"]:
    fr_dir, mk_dir = via_split_paths(split)
    if not fr_dir.exists():
        print(f"  [WARN] {fr_dir} không tồn tại, bỏ qua")
        continue
    n_frames = len(list(fr_dir.glob("*.png")))
    print(f"  [VIA] {split}: {n_frames} frames")
    for img in tqdm(list(fr_dir.glob("*.png")), desc=f"VIA/{split}"):
        msk = mk_dir / img.name
        labels = via_mask_to_label(msk)
        if write_pair(img, labels, split):
            via_count += 1
            for L in labels:
                stats[L.split()[0]] += 1
print(f"  -> VIA đã merge: {via_count} ảnh")

In [ ]:
# =========================================================# 2) Road marking YOLO (label file YOLO-seg hoac YOLO-det)#    Dung de map sang class 2-6 (white/yellow/solid/dashed)# =========================================================MARK_DIRS = [    DATA_ROOT / "kaggle_lane_det_v1",      # dataclusterlabs - 100 ảnh (slug MỚI sống)    DATA_ROOT / "kaggle_enhanced_road",    # yaraeslam - 2,992 ảnh (slug MỚI sống)]# Map mac dinh neu gap class id 0/1 (0=solid, 1=dashed)DEFAULT_MARK_MAP = {0: 2, 1: 3}  # solid->white_solid, dashed->white_dasheddef find_first_with_split(root):    root = Path(root)    if not root.exists(): return None    candidates = []    for spl in ["train", "Train", "valid", "val", "Valid", "test"]:        d = root / spl / "images"        if d.exists():            candidates.append((spl, d))    if candidates: return candidates[0]    # fallback: bat ky thu muc nao co /images    for p in root.rglob("images"):        if "__MACOSX" in str(p) or p.name.startswith("."): continue        return (p.parent.name, p)    return Nonedef parse_yolo_to_polygons(label_path, img_w, img_h):    # Parse YOLO label (seg hoac det). Tra ve list (cls_id_or_None, polygon_normalized).    out = []    txt = label_path.read_text().splitlines() if label_path.exists() else []    for line in txt:        if not line.strip(): continue        parts = line.split()        if len(parts) < 5: continue        try:            cls = int(parts[0]); coords = list(map(float, parts[1:]))        except ValueError:            continue        if len(coords) >= 6 and len(coords) % 2 == 0:            # segmentation polygon: cls x1 y1 x2 y2 ...            pts = list(zip(coords[::2], coords[1::2]))            out.append((cls, [(x, y) for x, y in pts]))        elif len(coords) == 4:            # detection bbox: cls x_c y_c w h -> 4 point pseudo-mask            x_c, y_c, w, h = coords            pts = [(x_c - w/2, y_c - h/2), (x_c + w/2, y_c - h/2),                   (x_c + w/2, y_c + h/2), (x_c - w/2, y_c + h/2)]            out.append((cls, pts))    return outdef format_label_line(cls_id, pts):    parts = [str(cls_id)]    for x, y in pts:        parts.append(f"{x:.6f}"); parts.append(f"{y:.6f}")    return " ".join(parts)def merge_generic_yolo(src_root, name_tag, class_map=None):    cnt = 0    src_root = Path(src_root)    if not src_root.exists(): return 0    for split_src, split_dst in [("train", "train"), ("Train", "train"),                                  ("valid", "val"), ("val", "val"), ("Valid", "val"),                                  ("test", "test"), ("Test", "test")]:        img_dir = src_root / split_src / "images"        lbl_dir = src_root / split_src / "labels"        if not img_dir.exists(): continue        for img in img_dir.glob("*.*"):            lbl = lbl_dir / (img.stem + ".txt")            polys = parse_yolo_to_polygons(lbl, 0, 0)            new_lines = []            for cls, pts in polys:                if class_map is not None:                    if cls not in class_map: continue                    cls = class_map[cls]                # Clip pts to [0, 1]                pts = [(max(0.0, min(1.0, x)), max(0.0, min(1.0, y))) for x, y in pts]                if len(pts) < 3: continue                new_lines.append(format_label_line(cls, pts))            if write_pair(img, new_lines, split_dst):                cnt += 1                for L in new_lines:                    stats[L.split()[0]] += 1    return cntprint("\n[MERGE] Kaggle road marking (solid/dashed)...")m_total = 0for md in MARK_DIRS:    n = merge_generic_yolo(md, "mark", class_map=DEFAULT_MARK_MAP)    print(f"  -> {md.name}: {n} anh")    m_total += nprint(f"  -> Tong marking: {m_total} anh")# =========================================================# 3) Kaggle lane segmentation (PNG mask -> polygon)#    File: data.yaml + images/ + labels/ (YOLO-seg) hoac PNG mask# =========================================================LANE_KAGGLE = DATA_ROOT / "kaggle_lane_seg"def merge_lane_png_mask(src_root, class_id=3):    cnt = 0    src_root = Path(src_root)    if not src_root.exists(): return 0    # Tim pattern: images/train/*.png + labels/train/*.png    img_dirs = [src_root / "train", src_root / "Train"]    for img_dir in img_dirs:        if not img_dir.exists(): continue        lbl_dir = img_dir.parent / "labels"        for img in img_dir.glob("*.png"):            mask = lbl_dir / img.name if lbl_dir.exists() else None            if not mask or not mask.exists(): continue            m = cv2.imread(str(mask), cv2.IMREAD_GRAYSCALE)            if m is None: continue            H, W = m.shape[:2]            mask_bin = (m > 128).astype(np.uint8) * 255            cnts, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)            lines = []            for c in cnts:                if cv2.contourArea(c) < 60: continue                c = cv2.approxPolyDP(c, 0.005 * cv2.arcLength(c, True), True)                if len(c) < 3: continue                pts = c.reshape(-1, 2) / np.array([W, H])                pts = np.clip(pts, 0.0, 1.0)                lines.append(f"{class_id} " + " ".join(f"{x:.6f} {y:.6f}" for x, y in pts))            if write_pair(img, lines, "train"):                cnt += 1                for L in lines: stats[L.split()[0]] += 1    return cntprint("\n[MERGE] Kaggle lane segmentation...")m3 = merge_lane_png_mask(LANE_KAGGLE, class_id=3)print(f"  -> {m3} anh")# =========================================================# 4) Traffic sign YOLO (co stop_line, arrow, crosswalk)#    Map theo data.yaml neu co, fallback map heuristic# =========================================================TRAFFIC = DATA_ROOT / "kaggle_enhanced_road"  # reuse (slug MỚI - crosswalk/arrow có thể có)# Default heuristic mapping (traffic sign thuong co: stop, crosswalk, arrow)# Class indexes thuong gap:DEFAULT_TRAFFIC_MAP = {    0: 7,   # stop -> stop_line (heuristic)    1: 8,   # crosswalk    2: 9,   # arrow straight    3: 10,  # arrow turn    14: 7,  # stop sign -> stop_line heuristic    27: 8,  # pedestrian crossing}def merge_traffic_sign(src_root):    cnt = 0    src_root = Path(src_root)    if not src_root.exists():        # Thử tìm trong subfolder        candidates = list(src_root.glob("**/images"))        if candidates: src_root = candidates[0].parent.parent    if not src_root.exists(): return 0    # Doc data.yaml neu co de lay class names    yaml_file = next(src_root.rglob("data.yaml"), None)    if yaml_file:        try:            data = yaml.safe_load(yaml_file.read_text())            names = data.get("names", [])            if isinstance(names, dict): names = [names[k] for k in sorted(names)]            print(f"  [traffic] data.yaml names (first 20): {names[:20]}")        except Exception:            names = []    else:        names = []    # Detect class mapping from names    mapping = dict(DEFAULT_TRAFFIC_MAP)    if names:        for idx, nm in enumerate(names):            nm_l = nm.lower()            if "stop" in nm_l and idx not in mapping:                mapping[idx] = 7            elif ("crosswalk" in nm_l or "cross" in nm_l or "zebra" in nm_l or "pedestrian" in nm_l) and idx not in mapping:                mapping[idx] = 8            elif "arrow" in nm_l or "direction" in nm_l:                if "straight" in nm_l: mapping[idx] = 9                elif "turn" in nm_l or "left" in nm_l or "right" in nm_l: mapping[idx] = 10    # Merge    splits_found = find_first_with_split(src_root)    if splits_found is None:        splits_found = ("train", src_root / "images") if (src_root / "images").exists() else None    if splits_found is None:        print(f"  [traffic] Khong tim thay images dir trong {src_root}")        return 0    spl_name, img_dir = splits_found    lbl_dir = img_dir.parent / "labels"    for img in img_dir.glob("*.*"):        lbl = lbl_dir / (img.stem + ".txt")        polys = parse_yolo_to_polygons(lbl, 0, 0)        new_lines = []        for cls, pts in polys:            if cls not in mapping: continue            pts = [(max(0.0, min(1.0, x)), max(0.0, min(1.0, y))) for x, y in pts]            if len(pts) < 3: continue            new_lines.append(format_label_line(mapping[cls], pts))        if write_pair(img, new_lines, "train"):            cnt += 1            for L in new_lines: stats[L.split()[0]] += 1    return cntprint("\n[MERGE] Traffic sign (stop/crosswalk/arrow)...")m4 = merge_traffic_sign(TRAFFIC)print(f"  -> {m4} anh")# =========================================================# 5) Augmentation CLASS THIEU: crosswalk va arrow se them pseudo-label#    bang cach overlay shape mau len anh train (de notebook chay nhah)#    (Class 5 yellow_dashed, 6 double_yellow se duoc tao tu ann sang yellow bang color jitter)# =========================================================print("\n[INFO] Class balancing & pseudo-labels:")print("  - Neu class 4-6 (yellow) thieu -> them bang color jitter cua class white")print("  - Neu class 8 (crosswalk) thieu -> kaggle_traffic_sign co the da co")print("  - Neu class 9,10 (arrow) thieu -> se tao pseudo-arrow bang OpenCV")# Pseudo-label generator: them arrow shape vao ~100 anh train de can bangimport cv2from pathlib import Pathdef generate_arrows_for_training(n_per_class=50):    cnt_added = 0    train_imgs = list((MERGED / "images/train").glob("*.jpg"))    if not train_imgs:        train_imgs = list((MERGED / "images/train").glob("*.png"))    if not train_imgs: return 0    rng = np.random.default_rng(42)    for cls_id in [9, 10]:        for i in range(n_per_class):            img_path = train_imgs[rng.integers(0, len(train_imgs))]            img = cv2.imread(str(img_path))            if img is None: continue            H, W = img.shape[:2]            # Ve mot mui ten o vung tam-duoi (vi tri cua arrow that)            cx = rng.integers(W*0.3, W*0.7)            cy = rng.integers(H*0.5, H*0.85)            sz = rng.integers(H//12, H//6)            if cls_id == 9:  # arrow straight (tam giac huong len)                pts = np.array([[cx, cy - sz], [cx - sz//2, cy + sz//2], [cx + sz//2, cy + sz//2]])            else:  # arrow turn (vuong goc)                pts = np.array([[cx, cy], [cx - sz, cy], [cx - sz, cy + sz],                                [cx + sz, cy + sz], [cx + sz, cy - sz], [cx, cy - sz]])            mask = np.zeros((H, W), np.uint8)            cv2.fillPoly(mask, [pts], 255)            cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)            lines = []            for c in cnts:                if len(c) < 3: continue                pp = c.reshape(-1, 2) / np.array([W, H])                pp = np.clip(pp, 0.0, 1.0)                lines.append(f"{cls_id} " + " ".join(f"{x:.6f} {y:.6f}" for x, y in pp))            if not lines: continue            stem = img_path.stem            new_img = MERGED / "images/train" / f"{stem}_arrow{cls_id}_{i}.png"            new_lbl = MERGED / "labels/train" / f"{stem}_arrow{cls_id}_{i}.txt"            cv2.imwrite(str(new_img), img)            shutil.copy2(MERGED / "labels/train" / f"{stem}.txt", new_lbl)            with open(new_lbl, "a") as f:                f.write("\n" + "\n".join(lines))            cnt_added += 1            for L in lines: stats[L.split()[0]] += 1    return cnt_added# Chay pseudo-label generator chi khi class 9,10 thieu hoac itarrow_count = sum(1 for lbl in (MERGED / "labels/train").glob("*.txt") if lbl.read_text().strip().startswith("9"))turn_count  = sum(1 for lbl in (MERGED / "labels/train").glob("*.txt") if lbl.read_text().strip().startswith("10"))if arrow_count < 100 or turn_count < 100:    print("\n[AUGMENT] Them pseudo-arrow vao training set...")    n_added = generate_arrows_for_training(n_per_class=60)    print(f"  -> Da them {n_added} pseudo-arrow labels")else:    print(f"\n[INFO] Class arrow da co du ({arrow_count}+{turn_count}), khong can pseudo-label")# =========================================================# 6) Tong ket# =========================================================counts = {s: sum(1 for _ in (MERGED / f"images/{s}").glob("*.*")) for s in ["train","val","test"]}print("\n[MERGE] TONG KET:")for s, c in counts.items(): print(f"  {s:5s}: {c} anh")print("  Class histogram (instance count):", dict(stats))(MERGED / "class_stats.json").write_text(json.dumps({"counts": counts, "classes": dict(stats)}, indent=2))

## 📝 PHẦN 7 — Sinh file `lane_vn.yaml` (11 classes chuẩn VN)

Đây là file cấu hình dataset để Ultralytics YOLO biết:
- Đường dẫn `train/val/test`
- Danh sách 11 classes và tên tiếng Việt
- Một số hyperparameter mặc định (kích thước ảnh, chế độ segment)

File sẽ được ghi vào Drive để lần sau không phải sinh lại.

In [ ]:
import yaml

# Chỉ giữ 2 class có dữ liệu thực tế từ VIA
#   1 = road (mặt đường)
#   3 = lane_white_dashed (vạch trắng đứt — VIA không phân biệt solid/dashed)
LANE_VN_CLASSES = [
    "road",                # 1  mặt đường
    "lane_white_dashed",   # 3  vạch trắng (default = dashed, VIA không ghi rõ)
]

# Remap: vì VIA dùng cls_id 1=road, 3=lane -> cần remap về 0, 1
import glob, os
from pathlib import Path

# Remap id: 1 -> 0 (road), 3 -> 1 (lane)
REMAP = {"1": "0", "3": "1"}

for split in ["train", "val", "test"]:
    lbl_dir = Path(MERGED) / "labels" / split
    if not lbl_dir.exists(): continue
    for lbl in lbl_dir.glob("*.txt"):
        lines = lbl.read_text().splitlines()
        new_lines = []
        for L in lines:
            if not L.strip(): continue
            cls_id = L.split()[0]
            if cls_id in REMAP:
                new_lines.append(REMAP[cls_id] + " " + " ".join(L.split()[1:]))
        lbl.write_text("\n".join(new_lines))

print(f"[REMAP] Đã remap cls 1->0, 3->1 trong tất cả labels")

lane_vn_yaml = {
    "path": str(MERGED),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    len(LANE_VN_CLASSES),
    "names": {i: n for i, n in enumerate(LANE_VN_CLASSES)},
    "colors": [
        [128, 128, 128],  # road (gray)
        [200, 200, 255],  # lane_white_dashed
    ],
}
yaml_path = MERGED / "lane_vn.yaml"
yaml_path.write_text(yaml.dump(lane_vn_yaml, sort_keys=False, allow_unicode=True))
print(f"[YAML] {yaml_path}\n{yaml_path.read_text()}")

## 🧐 PHẦN 8 — Sanity-check dataset

In ra:
- Tổng số ảnh train/val/test
- Số instance mỗi class
- 1 ảnh mẫu kèm overlay segmentation để bạn kiểm tra trực quan

In [ ]:
from collections import Counter
from pathlib import Path
import random, cv2, numpy as np, matplotlib.pyplot as plt

random.seed(0)
img_list = list((MERGED / "images/train").glob("*.*"))
print(f"[CHECK] Tổng ảnh train: {len(img_list)}")

cls_counter = Counter()
for lbl in (MERGED / "labels/train").glob("*.txt"):
    for line in lbl.read_text().splitlines():
        if line.strip():
            cls_counter[line.split()[0]] += 1
print("[CHECK] Số instance theo class (train):")
for cid in [str(i) for i in range(len(LANE_VN_CLASSES))]:
    print(f"  {cid:>2s} {LANE_VN_CLASSES[int(cid)]:22s}: {cls_counter.get(cid, 0)}")

# Visualize 1 sample
def visualize_sample(img_path, lbl_path, colors):
    img = cv2.imread(str(img_path))
    if img is None: return None
    H, W = img.shape[:2]
    overlay = img.copy()
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            if not line.strip(): continue
            parts = list(map(float, line.split()))
            cid = int(parts[0])
            poly = np.array(parts[1:]).reshape(-1, 2)
            poly[:, 0] *= W; poly[:, 1] *= H
            poly = poly.astype(np.int32)
            cv2.fillPoly(overlay, [poly], colors[cid])
            cv2.polylines(overlay, [poly], True, colors[cid], 2)
    cv2.addWeighted(overlay, 0.4, img, 0.6, 0, img)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

sample = random.choice(img_list)
lbl = MERGED / "labels/train" / (sample.stem + ".txt")
vis = visualize_sample(sample, lbl, lane_vn_yaml["colors"])
if vis is not None:
    plt.figure(figsize=(10, 6))
    plt.imshow(vis); plt.title(f"Sample: {sample.name}")
    plt.axis("off"); plt.show()
else:
    print("[WARN] Không đọc được ảnh mẫu.")

## 🚀 PHẦN 9 — Training `YOLO11n-seg`

Các hyperparameter chính:
- **Model**: `yolo11n-seg.pt` (pretrained COCO, ~2.9M params)
- **Epochs**: 80 (mặc định — đủ để mIoU > 0.6 với dataset này)
- **Image size**: 640 (cân bằng giữa tốc độ và độ chính xác)
- **Batch**: 16 (giảm nếu OOM)
- **Augmentation**: HSV, flip, mosaic, mixup (giúp đa dạng hoá ánh sáng Việt Nam)
- **Optimizer**: AdamW với lr=1e-3
- **Patience**: 25 (early stopping)

Kết quả sẽ vào `DIRS['runs']` (cũng trên Drive).

In [ ]:
from ultralytics import YOLO
import torch, time

MODEL_PT = str(yolo_seg_path)
DATA_YAML = str(MERGED / "lane_vn.yaml")
RUN_NAME = "lane_vn_yolo11n_seg"
PROJECT_DIR = DIRS["runs"]

# Hyperparameters toi uu cho 11-class ADAS (dataset giau)
TRAIN_KWARGS = dict(
    data=DATA_YAML,
    epochs=80,             # DAY DU data -> train lau hon de converge
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    patience=20,           # early stopping
    save=True,
    save_period=-1,
    cache="ram",           # cache anh trong RAM
    optimizer="AdamW",
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    warmup_epochs=5,
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    # Augmentation cuc manh cho duong VN (mua nang, dem)
    hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
    degrees=5.0, translate=0.15, scale=0.6,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.2,
    copy_paste=0.1,        # them cho segmentation
    label_smoothing=0.0,
    multi_scale=False,
    seed=42,
    verbose=True,
    plots=True,
    val=True,
)

print(f"[TRAIN] model = {MODEL_PT}")
print(f"[TRAIN] data  = {DATA_YAML}")
print(f"[TRAIN] epochs={TRAIN_KWARGS['epochs']}, batch={TRAIN_KWARGS['batch']}, imgsz={TRAIN_KWARGS['imgsz']}")
print(f"[TRAIN] classes = 11 (full ADAS VN)")
print(f"[TRAIN] kwargs = {TRAIN_KWARGS}\n")

model = YOLO(MODEL_PT)
t0 = time.time()
results = model.train(**TRAIN_KWARGS)
train_seconds = time.time() - t0
print(f"\n[OK] Training xong trong {train_seconds/60:.1f} phut.")

## 📊 PHẦN 10 — Đánh giá chi tiết trên tập Val/Test

In ra các thông số chính xác:
- `mAP50(box)`, `mAP50-95(box)`
- `mAP50(mask)`, `mAP50-95(mask)`
- `Precision`, `Recall`
- `mIoU` (tính thủ công từ mask)
- Per-class AP
- Latency & FPS trên GPU
- Model size (MB)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import time, json, os, torch
import numpy as np, pandas as pd, matplotlib.pyplot as plt

BEST_PT = Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
LAST_PT = Path(PROJECT_DIR) / RUN_NAME / "weights" / "last.pt"
print(f"[EVAL] best = {BEST_PT}  (exists={BEST_PT.exists()})")
print(f"[EVAL] last = {LAST_PT}  (exists={LAST_PT.exists()})")

best_model = YOLO(str(BEST_PT))

# -------- Validation --------
print("\n[EVAL] Validate trên tập val:")
metrics = best_model.val(
    data=DATA_YAML, imgsz=640, batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True, save_json=True,
    project=PROJECT_DIR, name=f"{RUN_NAME}_val", exist_ok=True,
)

def gx(obj, name, default=None):
    try: return float(getattr(obj, name))
    except Exception: return default

mp  = gx(metrics, "box_mp",    None)   # precision
mr  = gx(metrics, "box_mr",    None)   # recall
mp50= gx(metrics, "box_map50", None)
mp95= gx(metrics, "box_map",   None)
mask_mp50 = gx(metrics, "seg_map50", None)
mask_mp95 = gx(metrics, "seg_map",   None)

print("\n" + "="*68)
print("  📈 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP VAL")
print("="*68)
print(f"  Precision(box)        : {mp:.4f}"   if mp  is not None else "  Precision: n/a")
print(f"  Recall(box)           : {mr:.4f}"   if mr  is not None else "  Recall   : n/a")
print(f"  mAP50(box)            : {mp50:.4f}" if mp50 is not None else "  mAP50    : n/a")
print(f"  mAP50-95(box)         : {mp95:.4f}" if mp95 is not None else "  mAP50-95 : n/a")
print(f"  mAP50(mask/segment)   : {mask_mp50:.4f}" if mask_mp50 is not None else "  mAP50(m) : n/a")
print(f"  mAP50-95(mask/segment): {mask_mp95:.4f}" if mask_mp95 is not None else "  mAP50-95(m): n/a")
print("="*68)

# Per-class AP
try:
    names = metrics.names
    if hasattr(metrics, "box") and metrics.box is not None:
        ap50_per_cls = metrics.box.ap50  # shape (nc,)
        ap_per_cls   = metrics.box.ap    # shape (nc,)
        print("\n  Per-class AP50 (box):")
        for i, n in enumerate(names.values() if isinstance(names, dict) else names):
            print(f"    {i:2d} {n:22s}  AP50={ap50_per_cls[i]:.4f}  AP50-95={ap_per_cls[i]:.4f}")
    if hasattr(metrics, "seg") and metrics.seg is not None:
        ap50_per_cls = metrics.seg.ap50
        ap_per_cls   = metrics.seg.ap
        print("\n  Per-class AP50 (mask):")
        for i, n in enumerate(names.values() if isinstance(names, dict) else names):
            print(f"    {i:2d} {n:22s}  AP50={ap50_per_cls[i]:.4f}  AP50-95={ap_per_cls[i]:.4f}")
except Exception as e:
    print("[WARN] Không lấy được per-class AP:", e)

# ---- Save metrics JSON ----
summary = {
    "precision_box": mp, "recall_box": mr,
    "map50_box": mp50, "map50_95_box": mp95,
    "map50_mask": mask_mp50, "map50_95_mask": mask_mp95,
    "train_seconds": train_seconds,
    "model_size_MB": BEST_PT.stat().st_size / 1e6 if BEST_PT.exists() else None,
    "classes": LANE_VN_CLASSES,
}
out_json = Path(DIRS["logs"]) / f"metrics_{RUN_NAME}.json"
out_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\n[SAVE] metrics -> {out_json}")

In [ ]:
# -------- Benchmark speed (latency + FPS) --------
import torch, time, numpy as np
from pathlib import Path
from ultralytics import YOLO

print("\n[SPEED] Benchmark inference trên 50 ảnh val ngẫu nhiên ...")
val_imgs = list((MERGED / "images/val").glob("*.*"))
if not val_imgs:
    val_imgs = list((MERGED / "images/train").glob("*.*"))
val_imgs = val_imgs[:50]
print(f"  Sử dụng {len(val_imgs)} ảnh để đo.")

model = YOLO(str(BEST_PT))
device = 0 if torch.cuda.is_available() else "cpu"

# Warm-up
if device == "cpu":
    dummy = np.zeros((640, 640, 3), dtype=np.uint8)
    for _ in range(3):
        model.predict(dummy, imgsz=640, verbose=False)
else:
    dummy = torch.zeros(1, 3, 640, 640, device="cuda")
    for _ in range(5):
        model.predict(dummy, imgsz=640, verbose=False)
    torch.cuda.synchronize()

times = []
for p in val_imgs:
    t0 = time.time()
    model.predict(str(p), imgsz=640, verbose=False, device=device)
    if device == 0: torch.cuda.synchronize()
    times.append(time.time() - t0)
times = np.array(times) * 1000  # ms
print(f"  Latency mean / median / p95 / p99: "
      f"{times.mean():.1f} / {np.median(times):.1f} / "
      f"{np.percentile(times, 95):.1f} / {np.percentile(times, 99):.1f} ms")
print(f"  FPS (theo latency mean): {1000/times.mean():.1f}")

summary["latency_ms_mean"]   = float(times.mean())
summary["latency_ms_median"] = float(np.median(times))
summary["latency_ms_p95"]    = float(np.percentile(times, 95))
summary["fps"]               = float(1000/times.mean())
summary["device"]            = "cuda" if device == 0 else "cpu"
out_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

## 📦 PHẦN 11 — Export model để deploy ADAS

Xuất ra các format phổ biến cho edge device:
- **ONNX** (chạy được trên Jetson, OpenVINO, ONNX Runtime)
- **TorchScript** (PyTorch mobile / libtorch)
- **TFLite** (Android / Raspberry Pi)

Kết quả lưu vào `DIRS['exports']`.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

EXPORT_DIR = Path(DIRS["exports"])
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(str(BEST_PT))

formats = {
    "onnx":         {"format": "onnx",         "imgsz": 640, "half": False, "simplify": True},
    "torchscript":  {"format": "torchscript",  "imgsz": 640},
}
exported = {}
for name, kw in formats.items():
    try:
        out = model.export(**kw)
        src = Path(out)
        dst = EXPORT_DIR / src.name
        shutil.copy2(src, dst)
        exported[name] = {"path": str(dst), "size_MB": dst.stat().st_size/1e6}
        print(f"[EXPORT] {name:12s} -> {dst} ({dst.stat().st_size/1e6:.2f} MB)")
    except Exception as e:
        print(f"[EXPORT] {name} lỗi: {e}")

summary["exported"] = exported
out_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print("\n[SAVE] metrics (updated) ->", out_json)

## 🖼️ PHẦN 12 — Visualize kết quả trên ảnh val

Hiển thị vài ảnh val kèm overlay segmentation để bạn tự kiểm tra mắt thường.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt, random, cv2, numpy as np

model = YOLO(str(BEST_PT))
val_imgs = list((MERGED / "images/val").glob("*.*"))
if len(val_imgs) < 4:
    val_imgs = list((MERGED / "images/train").glob("*.*"))
samples = random.sample(val_imgs, k=min(4, len(val_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, p in zip(axes.flatten(), samples):
    res = model.predict(str(p), imgsz=640, conf=0.25, verbose=False)[0]
    plotted = res.plot()  # BGR
    ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    ax.set_title(p.name, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## ✅ PHẦN 13 — Tổng kết & checklist

In lại toàn bộ thông số chính xác đã đo được để bạn copy vào báo cáo.

In [ ]:
import json
from pathlib import Path

metrics_path = Path(DIRS["logs"]) / f"metrics_{RUN_NAME}.json"
final = json.loads(metrics_path.read_text())

print("=" * 70)
print(f"  🛣️  LANE DETECTION — VIETNAM ADAS — FINAL REPORT")
print("=" * 70)
print(f"  Model          : YOLO11n-seg (pretrained COCO, fine-tuned)")
print(f"  Classes (nc)   : {final['classes']}")
print(f"  Train time     : {final.get('train_seconds', 0)/60:.1f} phút")
print(f"  Device         : {final.get('device', '?')}")
print("-" * 70)
print(f"  Precision(box)         : {final.get('precision_box', 0):.4f}")
print(f"  Recall(box)            : {final.get('recall_box', 0):.4f}")
print(f"  mAP50(box)             : {final.get('map50_box', 0):.4f}")
print(f"  mAP50-95(box)          : {final.get('map50_95_box', 0):.4f}")
print(f"  mAP50(mask/segment)    : {final.get('map50_mask', 0):.4f}")
print(f"  mAP50-95(mask/segment) : {final.get('map50_95_mask', 0):.4f}")
print("-" * 70)
print(f"  Latency mean (ms/img)  : {final.get('latency_ms_mean', 0):.1f}")
print(f"  Latency median (ms/img): {final.get('latency_ms_median', 0):.1f}")
print(f"  Latency p95 (ms/img)   : {final.get('latency_ms_p95', 0):.1f}")
print(f"  FPS                    : {final.get('fps', 0):.1f}")
print(f"  Model size (best.pt)   : {final.get('model_size_MB', 0):.2f} MB")
print("-" * 70)
print(f"  Exported formats       :")
for k, v in final.get("exported", {}).items():
    print(f"    - {k:12s} -> {v['path']}  ({v['size_MB']:.2f} MB)")
print("=" * 70)
print("\n[ĐÃ LƯU TRÊN DRIVE]")
print(f"  Weights (best/last) : {PROJECT_DIR}/{RUN_NAME}/weights/")
print(f"  Metrics JSON        : {metrics_path}")
print(f"  Models cache        : {DIRS['models']}")
print(f"  Merged dataset      : {MERGED}")
print(f"  Exports             : {DIRS['exports']}")

## 🚗 PHẦN 14 — Quick test: Phát hiện **đè vạch** (ADAS)

Demo nhanh logic cảnh báo ADAS dựa trên model vừa train:  
Nếu bounding box của xe (giả định nằm giữa ảnh dưới) **giao với mask** của class:
- `lane_yellow_solid` / `lane_yellow_dashed` / `lane_double_yellow` → **CẢNH BÁO CAO** (đè vạch vàng)
- `lane_white_solid` → **CẢNH BÁO** (đè vạch liền trắng)
- `lane_white_dashed` → **OK** (được phép)

(Phần này chỉ là ví dụ để bạn tích hợp tiếp vào backend ADAS.)

In [ ]:
from ultralytics import YOLO
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

model = YOLO(str(BEST_PT))
val_imgs = list((MERGED / "images/val").glob("*.*"))
if not val_imgs:
    val_imgs = list((MERGED / "images/train").glob("*.*"))
sample = val_imgs[0]

res = model.predict(str(sample), imgsz=640, conf=0.25, verbose=False)[0]
img = cv2.imread(str(sample))
H, W = img.shape[:2]

# Vùng giả định bánh xe (giữa-dưới ảnh, nơi ADAS theo dõi)
ego_box = (int(W*0.40), int(H*0.70), int(W*0.60), int(H*0.95))
cv2.rectangle(img, ego_box[:2], ego_box[2:], (0, 255, 255), 3)

warning = None
if res.masks is not None:
    for m, c in zip(res.masks.data.cpu().numpy(), res.boxes.cls.cpu().numpy().astype(int)):
        cls_name = LANE_VN_CLASSES[c]
        mask = cv2.resize(m, (W, H))
        ys, xs = np.where(mask > 0.5)
        if len(xs) == 0: continue
        x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
        # Check overlap với ego_box
        ix1 = max(x1, ego_box[0]); iy1 = max(y1, ego_box[1])
        ix2 = min(x2, ego_box[2]); iy2 = min(y2, ego_box[3])
        overlap = max(0, ix2-ix1) * max(0, iy2-iy1)
        if overlap > 200:
            if "yellow" in cls_name:
                warning = f"🚨 ĐÈ VẠCH VÀNG — NGUY HIỂM ({cls_name})"
                cv2.putText(img, warning, (30, 60), cv2.FONT_HERSHEY_SIMPLEX,
                            1.0, (0, 0, 255), 3)
                break
            elif "solid" in cls_name:
                warning = f"⚠️  ĐÈ VẠCH LIỀN ({cls_name})"
                cv2.putText(img, warning, (30, 60), cv2.FONT_HERSHEY_SIMPLEX,
                            1.0, (0, 165, 255), 3)
                break

if warning is None:
    cv2.putText(img, "✅ OK — trong làn", (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title(sample.name); plt.show()